# MDPs in Python: Old Bessie, the Golden Goose, the Wavering Customer, Double or Nothing

Old Bessie 1 showed what all these games, and many other scenarios in life, have in common:

1. **States**: the situation you are in right now.
2. **Actions**: the choices you can make in a state, one at a time.
3. **Transition dynamics**: given a state and an action, they decide (1) the new state and (2) the reward.

And transition dynamics are usually random. Here you write all of it in Python, in the same order as the page.
Part 1 is the page's own example, Old Bessie WORN with the action RUN. Part 2 is three MDPs simpler than Old
Bessie: a goose that lays golden eggs, a customer who may leave, a quiz of double or nothing.

**How it works.** Run the cells from top to bottom and fill in the four 🎯 cells. In each of them the code is
already laid out for you: every `...` is one line, a `return` with the new state and the reward, and the comment
beside it says which row of the table it is. Every other cell is ready to run. A folded cell with a title holds
plumbing: run it, and open it only if you are curious. The 💡 hints are folded cells too: double-click one to
open it, and open the next one only if you are still stuck. It takes about 30 minutes.

Start with the two folded cells below: the kit and the checks.

In [ ]:
#@title The kit (run me first) { display-mode: "form" }
# The kit of the MDPs notebook. Run this cell first; you do not need to read it.
import inspect as _kit_inspect
import random as _kit_random

_KIT = {"luck": _kit_random.Random(6), "spins": 0, "forced": None, "page": []}


def spin():
    """The ball of a roulette: a number between 0 and 1, every number as likely as any other."""
    _KIT["spins"] = _KIT["spins"] + 1
    if _KIT["forced"] is not None:
        return _KIT["forced"]
    if _KIT["page"]:
        return _KIT["page"].pop(0)
    return _KIT["luck"].random()


def _kit_page_balls():
    """The 100 balls of page 1: Mulberry32 seeded 80, the generator of the arcade and of the pages, every
    second number (the first of each pair is the page's action die, unused there)."""
    state = 80
    balls = []
    for i in range(2 * 100):
        state = (state + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((state ^ (state >> 15)) * (1 | state)) & 0xFFFFFFFF
        t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
        if i % 2 == 1:
            balls.append(((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296)
    return balls


def page_1_dice():
    """The next 100 balls are the 100 rolls of Old Bessie page 1, in the page's order."""
    _KIT["page"] = _kit_page_balls()
    print("🎲 The next %d balls are the %d rolls of Old Bessie page 1." % (100, 100))


# The four MDPs, known by the names of their functions
_KIT_MDPS = {
    "bessie_transition": {
        "states": ["HEALTHY", "WORN", "SHAKY", "FAILING"], "actions": ["RUN", "SERVICE", "REPLACE"], "task": "A",
        "check": "check_bessie(bessie_transition)", "step": "week"},
    "goose_transition": {
        "states": ["GOOSE", "SOLD"], "actions": ["KEEP", "SELL"], "task": "B",
        "check": "check_goose(goose_transition)", "step": "week"},
    "customer_transition": {
        "states": ["LOYAL", "WAVERING", "GONE"], "actions": ["FULL_PRICE", "DISCOUNT"], "task": "C",
        "check": "check_customer(customer_transition)", "step": "week"},
    "quiz_transition": {
        "states": ["Q1", "Q2", "Q3", "Q4", "OVER"], "actions": ["ANSWER", "STOP"], "task": "D",
        "check": "check_quiz(quiz_transition)", "step": "question"},
}


def _kit_mdp(transition):
    return _KIT_MDPS.get(getattr(transition, "__name__", ""))


def _kit_is_int(x):
    return isinstance(x, int) and not isinstance(x, bool)


def _kit_state(mdp, s):
    if _kit_is_int(s) and 0 <= s < len(mdp["states"]):
        return mdp["states"][s]
    return repr(s)


def _kit_action(mdp, a):
    if _kit_is_int(a) and 0 <= a < len(mdp["actions"]):
        return mdp["actions"][a]
    return repr(a)


def _kit_money(x):
    """$10 for a whole number, $103.95 otherwise."""
    if isinstance(x, bool) or not isinstance(x, (int, float)):
        return repr(x)
    if x == int(x):
        return "$" + format(int(x), ",") if x >= 0 else "-$" + format(-int(x), ",")
    return "$" + format(x, ",.2f") if x >= 0 else "-$" + format(-x, ",.2f")


def _kit_gaps(fn):
    """How many lines of the function are still a gap: a line that starts with ..."""
    try:
        src = _kit_inspect.getsource(fn)
    except (OSError, TypeError):
        return 0
    n = 0
    for line in src.splitlines():
        if line.strip().startswith("..."):
            n = n + 1
    return n


def _kit_call(transition, state, action):
    """Calls the participant's function calmly: (new_state, reward), or None after one printed line."""
    mdp = _kit_mdp(transition)
    if mdp is None:
        print("⏳ give one of the four functions of the tasks: bessie_transition, goose_transition, "
              "customer_transition or quiz_transition.")
        return None
    name = transition.__name__
    if name != "bessie_transition":
        _KIT["page"] = []                       # page 1's dice belong to Old Bessie
    if _kit_gaps(transition) > 0:
        print("⏳ %s is not finished yet: fill in the 🎯 gaps of task %s first." % (name, mdp["task"]))
        return None
    call = "%s(%s, %s)" % (name, _kit_state(mdp, state), _kit_action(mdp, action))
    try:
        out = transition(state, action)
    except Exception as e:
        print("❌ %s stops with an error: %s: %s. %s says more." % (call, type(e).__name__, e, mdp["check"]))
        return None
    if not (isinstance(out, (tuple, list)) and len(out) == 2):
        print("❌ %s gives back %r: it should give back the new state and the reward. %s says more."
              % (call, out, mdp["check"]))
        return None
    return out[0], out[1]


def one_step(transition, state, action):
    """Calls transition(state, action) once and prints the new state and the reward, with their names."""
    out = _kit_call(transition, state, action)
    if out is None:
        return
    mdp = _kit_mdp(transition)
    print("state %-9s action %-11s ->  new state %-9s reward %s" % (
        _kit_state(mdp, state), _kit_action(mdp, action), _kit_state(mdp, out[0]), _kit_money(out[1])))


def many_times(transition, state, action, times=1000):
    """Calls transition(state, action) many times, a new ball each time, and counts what happens."""
    out = _kit_call(transition, state, action)          # the first call, calmly
    if out is None:
        return
    seen = []
    counts = []
    for i in range(times):
        if i > 0:
            try:
                out = transition(state, action)
                out = (out[0], out[1])
            except Exception:
                out = _kit_call(transition, state, action)
                if out is None:
                    return
        if out in seen:
            counts[seen.index(out)] = counts[seen.index(out)] + 1
        else:
            seen.append(out)
            counts.append(1)
    mdp = _kit_mdp(transition)
    print("state %s, action %s, %s times:" % (_kit_state(mdp, state), _kit_action(mdp, action), format(times, ",")))
    order = sorted(range(len(seen)), key=lambda i: (str(seen[i][0]), -counts[i]))
    for i in order:
        s, m = seen[i]
        print("   new state %-9s reward %-6s %6s times" % (_kit_state(mdp, s), _kit_money(m), format(counts[i], ",")))


print("✅ The kit is ready: spin, page_1_dice, one_step, many_times.")

In [ ]:
#@title The checks (run me too) { display-mode: "form" }
# The checks of the MDPs notebook. Run this cell too; you do not need to read it.

# Each test: state, action, ball, the new state and the reward the table gives, whether the ball decides, the row
_KIT_TESTS = {
    'bessie_transition': [
        (1, 0, 0.02, 3, -208, True, "a ball below 0.08 is a breakdown: the week's $72 minus the tow of $280, and the van ends FAILING"),
        (1, 0, 0.07, 3, -208, True, "a ball below 0.08 is a breakdown: the week's $72 minus the tow of $280, and the van ends FAILING"),
        (1, 0, 0.09, 1, 72, True, 'a ball from 0.08 to 0.356 is no wear: $72, and the van stays WORN'),
        (1, 0, 0.346, 1, 72, True, 'a ball from 0.08 to 0.356 is no wear: $72, and the van stays WORN'),
        (1, 0, 0.366, 2, 72, True, 'a ball from 0.356 to 0.862 wears the van one level: $72, and it is SHAKY'),
        (1, 0, 0.852, 2, 72, True, 'a ball from 0.356 to 0.862 wears the van one level: $72, and it is SHAKY'),
        (1, 0, 0.872, 3, 72, True, 'a ball of 0.862 or more wears the van two levels: $72, and it is FAILING'),
        (1, 0, 0.98, 3, 72, True, 'a ball of 0.862 or more wears the van two levels: $72, and it is FAILING'),
    ],
    'goose_transition': [
        (0, 0, 0.02, 0, 10, True, 'a ball below 0.8 brings a golden egg, $10, and the goose stays'),
        (0, 0, 0.79, 0, 10, True, 'a ball below 0.8 brings a golden egg, $10, and the goose stays'),
        (0, 0, 0.81, 0, 0, True, 'a ball of 0.8 or more brings no egg, $0, and the goose stays'),
        (0, 0, 0.98, 0, 0, True, 'a ball of 0.8 or more brings no egg, $0, and the goose stays'),
        (0, 1, 0.3, 1, 50, False, 'the butcher pays $50, and the goose is SOLD'),
        (0, 1, 0.95, 1, 50, False, 'the butcher pays $50, and the goose is SOLD'),
        (1, 0, 0.3, 1, 0, False, 'a SOLD goose brings nothing more and stays SOLD'),
        (1, 0, 0.95, 1, 0, False, 'a SOLD goose brings nothing more and stays SOLD'),
        (1, 1, 0.3, 1, 0, False, 'a SOLD goose brings nothing more and stays SOLD'),
        (1, 1, 0.95, 1, 0, False, 'a SOLD goose brings nothing more and stays SOLD'),
    ],
    'customer_transition': [
        (0, 0, 0.02, 1, 10, True, 'a LOYAL customer at FULL_PRICE pays $10 and starts to waver when the ball is below 0.3'),
        (0, 0, 0.29, 1, 10, True, 'a LOYAL customer at FULL_PRICE pays $10 and starts to waver when the ball is below 0.3'),
        (0, 0, 0.31, 0, 10, True, 'a LOYAL customer at FULL_PRICE pays $10 and stays LOYAL when the ball is 0.3 or more'),
        (0, 0, 0.98, 0, 10, True, 'a LOYAL customer at FULL_PRICE pays $10 and stays LOYAL when the ball is 0.3 or more'),
        (0, 1, 0.3, 0, 7, False, 'a DISCOUNT brings $7 and keeps the customer LOYAL'),
        (0, 1, 0.95, 0, 7, False, 'a DISCOUNT brings $7 and keeps the customer LOYAL'),
        (1, 0, 0.02, 2, 10, True, 'a WAVERING customer at FULL_PRICE pays $10 and leaves, GONE, when the ball is below 0.5'),
        (1, 0, 0.49, 2, 10, True, 'a WAVERING customer at FULL_PRICE pays $10 and leaves, GONE, when the ball is below 0.5'),
        (1, 0, 0.51, 1, 10, True, 'a WAVERING customer at FULL_PRICE pays $10 and keeps WAVERING when the ball is 0.5 or more'),
        (1, 0, 0.98, 1, 10, True, 'a WAVERING customer at FULL_PRICE pays $10 and keeps WAVERING when the ball is 0.5 or more'),
        (1, 1, 0.3, 0, 7, False, 'a DISCOUNT brings $7 and wins the customer back: LOYAL'),
        (1, 1, 0.95, 0, 7, False, 'a DISCOUNT brings $7 and wins the customer back: LOYAL'),
        (2, 0, 0.3, 2, 0, False, 'a customer who is GONE pays nothing and stays GONE'),
        (2, 0, 0.95, 2, 0, False, 'a customer who is GONE pays nothing and stays GONE'),
        (2, 1, 0.3, 2, 0, False, 'a customer who is GONE pays nothing and stays GONE'),
        (2, 1, 0.95, 2, 0, False, 'a customer who is GONE pays nothing and stays GONE'),
    ],
    'quiz_transition': [
        (0, 0, 0.02, 1, 0, True, 'a ball below 0.9 at Q1 is a right answer: on to Q2, no money yet'),
        (0, 0, 0.89, 1, 0, True, 'a ball below 0.9 at Q1 is a right answer: on to Q2, no money yet'),
        (0, 0, 0.91, 4, 0, True, 'a ball of 0.9 or more at Q1 is a wrong answer: nothing, and the game is OVER'),
        (0, 0, 0.98, 4, 0, True, 'a ball of 0.9 or more at Q1 is a wrong answer: nothing, and the game is OVER'),
        (0, 1, 0.3, 4, 0, False, 'STOP takes the bank of Q1 home, $0, and the game is OVER'),
        (0, 1, 0.95, 4, 0, False, 'STOP takes the bank of Q1 home, $0, and the game is OVER'),
        (1, 0, 0.02, 2, 0, True, 'a ball below 0.8 at Q2 is a right answer: on to Q3, no money yet'),
        (1, 0, 0.79, 2, 0, True, 'a ball below 0.8 at Q2 is a right answer: on to Q3, no money yet'),
        (1, 0, 0.81, 4, 0, True, 'a ball of 0.8 or more at Q2 is a wrong answer: nothing, and the game is OVER'),
        (1, 0, 0.98, 4, 0, True, 'a ball of 0.8 or more at Q2 is a wrong answer: nothing, and the game is OVER'),
        (1, 1, 0.3, 4, 100, False, 'STOP takes the bank of Q2 home, $100, and the game is OVER'),
        (1, 1, 0.95, 4, 100, False, 'STOP takes the bank of Q2 home, $100, and the game is OVER'),
        (2, 0, 0.02, 3, 0, True, 'a ball below 0.6 at Q3 is a right answer: on to Q4, no money yet'),
        (2, 0, 0.59, 3, 0, True, 'a ball below 0.6 at Q3 is a right answer: on to Q4, no money yet'),
        (2, 0, 0.61, 4, 0, True, 'a ball of 0.6 or more at Q3 is a wrong answer: nothing, and the game is OVER'),
        (2, 0, 0.98, 4, 0, True, 'a ball of 0.6 or more at Q3 is a wrong answer: nothing, and the game is OVER'),
        (2, 1, 0.3, 4, 200, False, 'STOP takes the bank of Q3 home, $200, and the game is OVER'),
        (2, 1, 0.95, 4, 200, False, 'STOP takes the bank of Q3 home, $200, and the game is OVER'),
        (3, 0, 0.02, 4, 800, True, 'a ball below 0.4 at Q4 is a right answer: $800, and the game is OVER'),
        (3, 0, 0.39, 4, 800, True, 'a ball below 0.4 at Q4 is a right answer: $800, and the game is OVER'),
        (3, 0, 0.41, 4, 0, True, 'a ball of 0.4 or more at Q4 is a wrong answer: nothing, and the game is OVER'),
        (3, 0, 0.98, 4, 0, True, 'a ball of 0.4 or more at Q4 is a wrong answer: nothing, and the game is OVER'),
        (3, 1, 0.3, 4, 400, False, 'STOP takes the bank of Q4 home, $400, and the game is OVER'),
        (3, 1, 0.95, 4, 400, False, 'STOP takes the bank of Q4 home, $400, and the game is OVER'),
        (4, 0, 0.3, 4, 0, False, 'when the game is OVER, nothing more happens'),
        (4, 0, 0.95, 4, 0, False, 'when the game is OVER, nothing more happens'),
        (4, 1, 0.3, 4, 0, False, 'when the game is OVER, nothing more happens'),
        (4, 1, 0.95, 4, 0, False, 'when the game is OVER, nothing more happens'),
    ],
}


def _kit_check(key, transition):
    """Runs the tests of one MDP and prints one line. True when all pass."""
    mdp = _KIT_MDPS[key]
    if not callable(transition):
        print("⏳ %s is not a function yet: run the cell of task %s first." % (key, mdp["task"]))
        return False
    gaps = _kit_gaps(transition)
    if gaps > 0:
        print("⏳ %s is not finished yet: replace every ... of task %s with your code (%d left)."
              % (key, mdp["task"], gaps))
        return False
    tests = _KIT_TESTS[key]
    for state, action, ball, want_state, want_reward, decides, why in tests:
        call = "%s(%s, %s)" % (key, _kit_state(mdp, state), _kit_action(mdp, action))
        if decides:
            call = call + " with the ball %s" % ball
        _KIT["forced"] = ball
        _KIT["spins"] = 0
        try:
            out = transition(state, action)
        except Exception as e:
            print("❌ %s stops with an error: %s: %s" % (call, type(e).__name__, e))
            return False
        finally:
            _KIT["forced"] = None
        spins = _KIT["spins"]
        want = "(%s, %s)" % (_kit_state(mdp, want_state), _kit_money(want_reward))
        if out is None:
            print("❌ %s gives back nothing (None). Every 🎯 line becomes a return line with two values, the new "
                  "state and the reward. The table says %s: %s." % (call, want, why))
            return False
        if not (isinstance(out, (tuple, list)) and len(out) == 2):
            print("❌ %s gives back %r. It should give back two values, the new state and the reward: the table "
                  "says %s, so return %s, %s." % (call, out, want, mdp["states"][want_state], want_reward))
            return False
        if isinstance(out[0], str):
            print("❌ %s gives back %r. A state is a number: write %s without quotes."
                  % (call, tuple(out), out[0]))
            return False
        if spins != 1:
            print("❌ %s spins %d balls in one %s. A %s has one ball: keep the first line, ball = spin(), and "
                  "use ball below it." % (key, spins, mdp["step"], mdp["step"]))
            return False
        if out[0] != want_state or out[1] != want_reward:
            print("❌ %s gives back (%s, %s). The table says %s: %s."
                  % (call, _kit_state(mdp, out[0]), _kit_money(out[1]), want, why))
            return False
    print("✅ %s follows the table in all %d tests. Task %s is done." % (key, len(tests), mdp["task"]))
    return True


def check_bessie(bessie_transition):
    _kit_check("bessie_transition", bessie_transition)


def check_goose(goose_transition):
    _kit_check("goose_transition", goose_transition)


def check_customer(customer_transition):
    _kit_check("customer_transition", customer_transition)


def check_quiz(quiz_transition):
    _kit_check("quiz_transition", quiz_transition)


print("✅ The checks are ready: check_bessie, check_goose, check_customer, check_quiz.")

# Part 1. Old Bessie

## States

A state is the situation you are in right now. It holds what you need to know to choose what to do next. Old
Bessie's state is how worn the van is: HEALTHY, WORN, SHAKY or FAILING. In Python a state is a number, with a name
that reads like the story.

In [ ]:
HEALTHY, WORN, SHAKY, FAILING = 0, 1, 2, 3      # the states: how worn the van is

print("Old Bessie's states:", HEALTHY, WORN, SHAKY, FAILING)

## Actions

An action is a choice you can make in a state. You choose one action at a time. In every state, Old Bessie's
owner chooses one of three actions: RUN (drive the route this week), SERVICE (a week in the shop) or REPLACE (buy
a new van). Actions are numbers too.

In [ ]:
RUN, SERVICE, REPLACE = 0, 1, 2                 # the actions

print("Old Bessie's actions:", RUN, SERVICE, REPLACE)

## Transition dynamics

Given a state and an action, the transition dynamics decide (1) the new state and (2) the reward. In Python they
are one function: it takes a state and an action, and a line such as `return SHAKY, 72` gives back both, first
the new state, then the reward.

On the page a die decided what happens. In Python the luck comes from `spin()`, the kit's roulette ball: a number
between 0 and 1, every number as likely as any other. Something that happens with probability 0.8 happens when
the ball lands below 0.8.

💭 **Think first.** Of 1,000 balls, how many land below 0.8?

In [ ]:
ball = spin()
print("one ball:", ball)

below = 0
for i in range(1000):
    if spin() < 0.8:
        below = below + 1
print(below, "of 1,000 balls landed below 0.8")

**Reading the result.** About 800, a few more or fewer each time you run the cell. So `if ball < 0.8:` is how
Python says "with probability 0.8".

Page 1's example is the state WORN and the action RUN. The van earns its week, \$72, and then one of four things
happens:

| State | Action | Probability | New state | Reward | What happens |
|---|---|---|---|---|---|
| WORN | RUN | 0.08 | FAILING | -\$208 | a breakdown: \$72 minus the tow of \$280 |
| WORN | RUN | 0.276 | WORN | \$72 | no wear |
| WORN | RUN | 0.506 | SHAKY | \$72 | wears one level |
| WORN | RUN | 0.138 | FAILING | \$72 | wears two levels |

One ball decides the row. The rows take turns along the line from 0 to 1: the breakdown takes the balls below
0.08, no wear the next 0.276 of the line, up to 0.08 + 0.276 = 0.356, one level up to 0.356 + 0.506 = 0.862, and
two levels the rest. These running sums are already in the code.

## 🎯 Task A. Old Bessie's transition dynamics, WORN and RUN

Fill in `bessie_transition`: each `...` becomes the `return` of its row. The first row, the breakdown, is written
for you.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# Each 🎯 line becomes one `return`: the new state of its row, then the reward.
# No wear keeps the van WORN and brings $72: return WORN, 72

In [ ]:
#@title 💡 Hint 2 for task A: the three lines (double-click to open) { display-mode: "form" }
#             return WORN, 72
#             return SHAKY, 72
#         return FAILING, 72

In [ ]:
def bessie_transition(state, action):
    """Old Bessie's transition dynamics for page 1's example, WORN and RUN: gives back the new state and the reward."""
    ball = spin()                          # the week's ball: a number between 0 and 1
    if state == WORN and action == RUN:
        if ball < 0.08:
            return FAILING, 72 - 280       # probability 0.08: a breakdown, the week's $72 minus the tow of $280
        if ball < 0.356:
            return WORN, 72                # probability 0.276: no wear, $72, the van stays WORN
        if ball < 0.862:
            return SHAKY, 72               # probability 0.506: wears one level, $72, SHAKY
        return FAILING, 72                 # probability 0.138: wears two levels, $72, FAILING

In [ ]:
check_bessie(bessie_transition)

## One step

Your function is the transition dynamics of WORN and RUN. `one_step` calls it once and prints what it gives back,
with the names. Run the cell a few times: each run spins a new ball.

In [ ]:
one_step(bessie_transition, WORN, RUN)

## Transition dynamics are usually random

For the same state and the same action, the new state and the reward can be different every time.
`page_1_dice()` hands your function the balls of page 1, so the six rolls below are the page's six rolls.

In [ ]:
page_1_dice()                                   # the balls of page 1
for roll in range(6):
    one_step(bessie_transition, WORN, RUN)

**Reading the result.** Six times WORN and RUN, and four different outcomes: SHAKY, WORN, FAILING after two
levels of wear, and in the sixth roll a breakdown, -\$208. The same six as on the page.

💭 **Think first.** The page then counts all its outcomes in 100 rolls. How many of the 100 are breakdowns?

In [ ]:
page_1_dice()                                   # the balls of page 1 again, from its first roll
many_times(bessie_transition, WORN, RUN, times=100)

**Reading the result.** 52 SHAKY, 27 WORN, 15 FAILING after two levels of wear and 6 breakdowns: the counts of
page 1. They follow the probabilities of the table, 0.506, 0.276, 0.138 and 0.08, with some luck. With more rolls
the counts come closer: here are 10,000, on the notebook's own luck.

In [ ]:
many_times(bessie_transition, WORN, RUN, times=10000)

**Reading the result.** About 5,060 SHAKY, 2,760 WORN, 1,380 FAILING after wear and 800 breakdowns: 0.506, 0.276,
0.138 and 0.08 of 10,000.

# Part 2. Three MDPs simpler than Old Bessie

Each of them in the same steps as Old Bessie: the states, the actions, the transition dynamics, and how random
they are.

## The Golden Goose

A farmer owns a goose that lays a golden egg, worth \$10, in 80% of the weeks. A butcher offers \$50 for the
goose. A sold goose is gone for good.

### States and actions

Two states: the farmer still has the GOOSE, or it is SOLD. Two actions, chosen every week: KEEP the goose, or
SELL it.

In [ ]:
GOOSE, SOLD = 0, 1          # the states
KEEP, SELL = 0, 1           # the actions

### Transition dynamics

| State | Action | Probability | New state | Reward |
|---|---|---|---|---|
| GOOSE | KEEP | 0.8 | GOOSE | \$10 |
| GOOSE | KEEP | 0.2 | GOOSE | \$0 |
| GOOSE | SELL | 1 | SOLD | \$50 |
| SOLD | KEEP or SELL | 1 | SOLD | \$0 |

This table is the whole MDP: two states, two actions, and the transition dynamics, one row per outcome.

## 🎯 Task B. The goose's transition dynamics

Fill in `goose_transition`: one block per state and action, as in the table. A row with probability 1 is a single
`return`; two rows share the ball, the first takes the balls below its probability, the second the rest.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# Each 🎯 line becomes one `return`, and each `return` is one row of the table: the new state, then the reward.
# The row of SELL has probability 1, the new state SOLD and the reward $50.

In [ ]:
#@title 💡 Hint 2 for task B: the three lines (double-click to open) { display-mode: "form" }
#             return GOOSE, 10
#         return GOOSE, 0
#         return SOLD, 50

In [ ]:
def goose_transition(state, action):
    """The goose's transition dynamics: gives back the new state and the reward of one week."""
    ball = spin()                          # the week's ball
    if state == GOOSE and action == KEEP:
        if ball < 0.8:
            return GOOSE, 10               # probability 0.8: a golden egg, $10, the goose stays
        return GOOSE, 0                    # probability 0.2: no egg, $0, the goose stays
    if state == GOOSE and action == SELL:
        return SOLD, 50                    # probability 1: the butcher pays $50, the goose is SOLD
    if state == SOLD:
        return SOLD, 0                     # probability 1, whatever the action: a sold goose brings nothing more

In [ ]:
check_goose(goose_transition)

### Transition dynamics are usually random

💭 **Think first.** The farmer keeps the goose in 1,000 separate weeks, each with a new ball. In how many of them
is there an egg?

In [ ]:
many_times(goose_transition, GOOSE, KEEP)

**Reading the result.** About 800 eggs and 200 weeks without one: the counts follow the probabilities of the
table. Try `GOOSE, SELL`: probability 1, so the same outcome every time.

## The Wavering Customer

A customer buys from you every week. At FULL_PRICE they pay \$10, but a LOYAL customer starts to waver in 30% of
the weeks, and a WAVERING customer leaves for good in 50% of the weeks, after paying. A DISCOUNT brings \$7 and
makes the customer LOYAL next week.

### States and actions

Three states: the customer is LOYAL, WAVERING, or GONE. Two actions, chosen every week: FULL_PRICE or DISCOUNT.

In [ ]:
LOYAL, WAVERING, GONE = 0, 1, 2       # the states
FULL_PRICE, DISCOUNT = 0, 1           # the actions

### Transition dynamics

| State | Action | Probability | New state | Reward |
|---|---|---|---|---|
| LOYAL | FULL_PRICE | 0.3 | WAVERING | \$10 |
| LOYAL | FULL_PRICE | 0.7 | LOYAL | \$10 |
| LOYAL | DISCOUNT | 1 | LOYAL | \$7 |
| WAVERING | FULL_PRICE | 0.5 | GONE | \$10 |
| WAVERING | FULL_PRICE | 0.5 | WAVERING | \$10 |
| WAVERING | DISCOUNT | 1 | LOYAL | \$7 |
| GONE | FULL_PRICE or DISCOUNT | 1 | GONE | \$0 |

## 🎯 Task C. The customer's transition dynamics

Fill in `customer_transition`. The first block, LOYAL and FULL_PRICE, is written for you: its two rows, 0.3 and
0.7, share the ball as in task B.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# LOYAL and DISCOUNT is one row with probability 1: the new state LOYAL and the reward 7.
# WAVERING and FULL_PRICE is two rows: below 0.5 the customer pays $10 and is GONE, otherwise pays $10 and
# stays WAVERING.

In [ ]:
#@title 💡 Hint 2 for task C: the five lines (double-click to open) { display-mode: "form" }
#         return LOYAL, 7
#             return GONE, 10
#         return WAVERING, 10
#         return LOYAL, 7
#         return GONE, 0

In [ ]:
def customer_transition(state, action):
    """The customer's transition dynamics: gives back the new state and the reward of one week."""
    ball = spin()                          # the week's ball
    if state == LOYAL and action == FULL_PRICE:
        if ball < 0.3:
            return WAVERING, 10            # probability 0.3: pays $10, then starts to waver
        return LOYAL, 10                   # probability 0.7: pays $10 and stays LOYAL
    if state == LOYAL and action == DISCOUNT:
        return LOYAL, 7                    # probability 1: pays $7 and stays LOYAL
    if state == WAVERING and action == FULL_PRICE:
        if ball < 0.5:
            return GONE, 10                # probability 0.5: pays $10 one last time, then GONE
        return WAVERING, 10                # probability 0.5: pays $10 and keeps WAVERING
    if state == WAVERING and action == DISCOUNT:
        return LOYAL, 7                    # probability 1: pays $7 and is won back, LOYAL
    if state == GONE:
        return GONE, 0                     # probability 1, whatever the action: GONE, $0

In [ ]:
check_customer(customer_transition)

### Transition dynamics are usually random

💭 **Think first.** A WAVERING customer meets FULL_PRICE in 1,000 separate weeks, each with a new ball. In how many
of them does the customer leave?

In [ ]:
many_times(customer_transition, WAVERING, FULL_PRICE)

**Reading the result.** About 500 GONE and 500 still WAVERING. Try another state and action, such as
`LOYAL, FULL_PRICE`.

## Double or Nothing

A quiz of four questions, Q1 to Q4. Before each question you STOP and take the bank home, or you ANSWER. The bank
is \$0 before Q1, \$100 before Q2, \$200 before Q3 and \$400 before Q4, and a right answer at Q4 wins \$800. A wrong
answer ends the game with nothing. The questions get harder: you answer right with probability 0.9, 0.8, 0.6 and
0.4.

### States and actions

Five states: the question you face, Q1 to Q4, or OVER. Two actions: ANSWER or STOP. The four questions follow the
same rules with other numbers, so two lists hold the numbers, and the state, itself a number, picks from them:
`BANK[Q3]` is the bank at Q3.

In [ ]:
Q1, Q2, Q3, Q4, OVER = 0, 1, 2, 3, 4      # the states: the question you face, or the game is over
ANSWER, STOP = 0, 1                        # the actions
BANK = [0, 100, 200, 400]                  # what STOP takes home at Q1, Q2, Q3, Q4
PROBABILITY = [0.9, 0.8, 0.6, 0.4]         # the probability of a right answer at Q1, Q2, Q3, Q4

print("At Q3 the bank is", BANK[Q3], "and a right answer has probability", PROBABILITY[Q3])

### Transition dynamics

| State | Action | Probability | New state | Reward |
|---|---|---|---|---|
| Q1, Q2, Q3, Q4 | STOP | 1 | OVER | the bank: \$0, \$100, \$200, \$400 |
| Q1, Q2, Q3 | ANSWER | right: 0.9, 0.8, 0.6 | the next question | \$0 |
| Q4 | ANSWER | right: 0.4 | OVER | \$800 |
| Q1, Q2, Q3, Q4 | ANSWER | wrong: 0.1, 0.2, 0.4, 0.6 | OVER | \$0 |
| OVER | ANSWER or STOP | 1 | OVER | \$0 |

## 🎯 Task D. The quiz's transition dynamics

Fill in `quiz_transition`. `BANK[state]` is the bank of the question you face, and `state + 1` is the next
question: `Q3 + 1` is `Q4`.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# STOP: the new state is OVER, and the reward is the bank of this question, BANK[state].
# A right answer at Q4 gives OVER and 800; at another question, the next question, state + 1, and 0.
# A wrong answer gives OVER and 0.

In [ ]:
#@title 💡 Hint 2 for task D: the four lines (double-click to open) { display-mode: "form" }
#         return OVER, BANK[state]
#             return OVER, 800
#         return state + 1, 0
#     return OVER, 0

In [ ]:
def quiz_transition(state, action):
    """The quiz's transition dynamics: gives back the new state and the reward, the money taken home."""
    ball = spin()                          # the question's ball
    if state == OVER:
        return OVER, 0                     # probability 1: the game is over, nothing more happens
    if action == STOP:
        return OVER, BANK[state]           # probability 1: OVER, and the bank of this question
    if ball < PROBABILITY[state]:          # a right answer
        if state == Q4:
            return OVER, 800               # right at the last question: OVER, $800
        return state + 1, 0                # right at Q1, Q2 or Q3: the next question, $0
    return OVER, 0                         # a wrong answer: OVER, $0

In [ ]:
check_quiz(quiz_transition)

### Transition dynamics are usually random

💭 **Think first.** You ANSWER at Q3 in 1,000 separate games. How often do you reach Q4?

In [ ]:
many_times(quiz_transition, Q3, ANSWER)

**Reading the result.** About 600 times Q4 and 400 times OVER with nothing: the probability 0.6 of a right answer
at Q3.

## What the four MDPs have in common

| MDP | States | Actions | Transition dynamics |
|---|---|---|---|
| Old Bessie | HEALTHY, WORN, SHAKY, FAILING | RUN, SERVICE, REPLACE | `bessie_transition` (WORN and RUN) |
| The Golden Goose | GOOSE, SOLD | KEEP, SELL | `goose_transition` |
| The Wavering Customer | LOYAL, WAVERING, GONE | FULL_PRICE, DISCOUNT | `customer_transition` |
| Double or Nothing | Q1, Q2, Q3, Q4, OVER | ANSWER, STOP | `quiz_transition` |

Every one of them is states, actions and transition dynamics, and every transition function has the same shape:
one ball, then one `return` per row of the table, the new state and the reward. Next, Old Bessie 2 lets the steps
follow one another: episodes and returns.

Try any MDP, state and action in the cell below.

In [ ]:
many_times(quiz_transition, Q4, ANSWER)          # change me: any MDP, state and action